# 03 · BERT Classification

Fine-tune `bert-base-uncased` with two classification heads (UVP and Data Uniqueness) using 5-fold cross-validation and class weights (thesis §3.4).

Input: the annotated evaluation set `data/processed/data_companies.csv`.

In [ ]:
import pandas as pd

from company_assessment import config
from company_assessment.models.bert import prepare_companies, cross_validate_model
from company_assessment.evaluation import (
    majority_class_accuracy,
    random_predictor_accuracy,
    plot_result_matrices,
)

### Load and clean the annotated data

In [ ]:
companies = pd.read_csv(config.ANNOTATED_CSV, na_values=[''], keep_default_na=False)
companies['Data Uniqueness'] = companies['Data Uniqueness'].fillna('N/A')
companies = companies.dropna(subset=['UVP']).reset_index(drop=True)
companies.shape

### Baseline accuracies

Majority-class and frequency-weighted random baselines to compare the model against.

In [ ]:
companies = prepare_companies(companies)

for name, col in [('UVP', 'uvp_label'), ('Data Uniqueness', 'db_label')]:
    print(f'{name:>16} | majority {majority_class_accuracy(companies[col]):.2f} | '
          f'random {random_predictor_accuracy(companies[col]):.2f}')

### Cross-validate the model

> Requires PyTorch + a downloaded `bert-base-uncased`; use a GPU if available.

In [ ]:
results = cross_validate_model(companies)

### Confusion matrices

In [ ]:
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plot_result_matrices(results, save_dir=config.FIGURES_DIR)